# TetheredAI MLB Model Lab — No-Lineup Feature Set

This notebook is the manual **model lab** for MLB moneyline predictions. It reads the committed project data, performs EDA, compares feature sets/models, tunes hyperparameters, evaluates calibration, tunes edge thresholds when odds history exists, and exports a champion `.joblib` model for GitHub Actions scoring.

Design choices:
- No lineup features.
- Pregame-safe rolling/expanding features only.
- Daily GitHub Action should score with the saved champion model, not retrain.
- Model selection should prioritize log loss, Brier score, calibration, AUC, and betting-edge stability over plain accuracy.


In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, log_loss, brier_score_loss
from sklearn.inspection import permutation_importance

# Find project root whether running from notebooks/ or project root.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'src').exists() and (parent / 'data').exists():
        ROOT = parent
        break

sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

from mlb_betting.config import get_settings
from mlb_betting.db import connect
from mlb_betting.feature_engineering import get_model_feature_columns
from mlb_betting.modeling import (
    build_feature_sets,
    compare_models,
    probability_buckets,
    export_champion_model,
    tune_moneyline_edge_thresholds,
    add_moneyline_edges,
)


## 1. Load data

The feature file should be created by `scripts/03_build_features.py`. It is sourced from `data/odds.db` tables including MLB schedule/results, pitcher boxscores, team boxscores, and odds snapshots.

In [ ]:
settings = get_settings()
FEATURE_PATH = settings.data_dir / 'processed' / 'mlb_game_features.parquet'
PRED_PATH = settings.data_dir / 'predictions' / 'mlb_moneyline_predictions.csv'
DB_PATH = settings.odds_db_path

print('Feature path:', FEATURE_PATH, FEATURE_PATH.exists())
print('Predictions path:', PRED_PATH, PRED_PATH.exists())
print('DB path:', DB_PATH, DB_PATH.exists())

features = pd.read_parquet(FEATURE_PATH)
features['game_datetime_utc'] = pd.to_datetime(features['game_datetime_utc'], utc=True, errors='coerce')
print(features.shape)
display(features.head())


## 2. Dataset QA

In [ ]:
completed = features[features['target_home_win'].notna()].copy()
upcoming = features[features['target_home_win'].isna()].copy()
print('Completed games:', len(completed))
print('Upcoming games:', len(upcoming))
print('Date range:', features['game_datetime_utc'].min(), '→', features['game_datetime_utc'].max())
print('Home win rate:', completed['target_home_win'].mean())

qa_cols = ['game_pk','official_date','game_datetime_utc','home_team_name','away_team_name','target_home_win','home_score','away_score']
display(features[[c for c in qa_cols if c in features.columns]].tail(20))


In [ ]:
def summarize_missingness(df):
    out = pd.DataFrame({
        'column': df.columns,
        'missing_count': df.isna().sum().values,
        'missing_pct': df.isna().mean().values,
        'non_null_count': df.notna().sum().values,
    }).sort_values(['missing_pct','missing_count'], ascending=[False, False])
    return out

feature_cols_all = get_model_feature_columns(completed, include_market=False, min_non_null_rate=0.00)
miss = summarize_missingness(completed[feature_cols_all])
display(miss.head(40))


## 3. Feature group audit

This verifies that the enhanced no-lineup feature groups are present: Elo, starter, bullpen, team boxscore, and team-vs-handedness.

In [ ]:
groups = {
    'elo': [c for c in features.columns if 'elo' in c.lower()],
    'starter': [c for c in features.columns if 'starter_' in c.lower()],
    'bullpen': [c for c in features.columns if 'bullpen_' in c.lower()],
    'team_box': [c for c in features.columns if 'team_box_' in c.lower()],
    'team_vs_hand': [c for c in features.columns if 'team_vs_hand' in c.lower()],
    'market': [c for c in features.columns if any(k in c.lower() for k in ['market_', 'moneyline', 'spread', 'total_points', 'vig', 'book_count'])],
}
for name, cols in groups.items():
    print(f'{name}: {len(cols)}')
    print(cols[:12])


## 4. Baselines

Always compare against simple baselines. A model is not useful unless it beats these on probabilistic metrics, not just accuracy.

In [ ]:
y = completed['target_home_win'].astype(int).to_numpy()
home_rate = y.mean()
base_prob = np.repeat(home_rate, len(y))
print('Home win rate baseline:', home_rate)
print('Baseline log_loss:', log_loss(y, base_prob))
print('Baseline brier:', brier_score_loss(y, base_prob))


## 5. Build feature sets

No market features are used for the pure baseball model. Market columns stay in the feature file for edge calculation and recommendations.

In [ ]:
feature_sets = build_feature_sets(completed, min_non_null_rate=0.05)
for name, cols in feature_sets.items():
    print(name, len(cols))


## 6. Tune and compare models

This includes logistic regression, random forest, extra trees, histogram gradient boosting, SVM, and optionally XGBoost/LightGBM if installed.

Set `RUN_FULL_SEARCH = False` for fast iteration; set it to `True` for a deeper monthly/manual model run.

In [ ]:
RUN_FULL_SEARCH = False
MODEL_NAMES = None  # Example: ['random_forest','lightgbm','xgboost','svm_rbf']

comparison = compare_models(
    completed,
    feature_sets=feature_sets,
    model_names=MODEL_NAMES,
    holdout_days=60,
    tune=True,
    calibrate=True,
    max_search_iter=80 if RUN_FULL_SEARCH else 25,
    max_cv_splits=4,
)
results = comparison['results'].copy()
valid_results = results[results['holdout_log_loss'].notna()].sort_values(['holdout_log_loss','holdout_brier'])
display(valid_results[[
    'candidate_key','feature_set','model_name','feature_count','calibration',
    'holdout_log_loss','holdout_brier','holdout_roc_auc','holdout_accuracy_50pct','holdout_avg_pred','holdout_actual_rate','best_params'
]].head(25))
print('Best key:', comparison['best_key'])


In [ ]:
if len(valid_results):
    plot_df = valid_results.head(15).copy().sort_values('holdout_log_loss')
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df['candidate_key'], plot_df['holdout_log_loss'])
    plt.xlabel('Holdout log loss - lower is better')
    plt.title('Top candidate models by log loss')
    plt.gca().invert_yaxis()
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.barh(plot_df['candidate_key'], plot_df['holdout_brier'])
    plt.xlabel('Holdout Brier score - lower is better')
    plt.title('Top candidate models by Brier score')
    plt.gca().invert_yaxis()
    plt.show()


## 7. Best model diagnostics

In [ ]:
best_key = comparison['best_key']
best = comparison['fitted'][best_key]
test = comparison['test'].copy()
holdout_preds = best['holdout_predictions'].copy()

y_true = holdout_preds['target_home_win'].astype(int)
y_prob = holdout_preds['model_home_win_prob'].astype(float)
y_pred = (y_prob >= 0.5).astype(int)

print('Best model:', best_key)
print('Feature count:', len(best['feature_cols']))
print('Metrics:', best['metrics'])
print('Best params:', best['best_params'])
print(classification_report(y_true, y_pred, digits=3))
display(pd.DataFrame(confusion_matrix(y_true, y_pred), index=['actual_away','actual_home'], columns=['pred_away','pred_home']))

fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'AUC = {auc(fpr,tpr):.3f}')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend()
plt.show()

display(probability_buckets(holdout_preds, 'model_home_win_prob', target_col='target_home_win'))


## 8. Feature importance

Permutation importance can be slow. For fast iteration, reduce `n_repeats`.

In [ ]:
RUN_PERMUTATION_IMPORTANCE = True
if RUN_PERMUTATION_IMPORTANCE:
    X_test = comparison['test'][best['feature_cols']]
    y_test = comparison['test']['target_home_win'].astype(int)
    imp = permutation_importance(
        best['estimator'], X_test, y_test,
        n_repeats=5,
        random_state=42,
        scoring='neg_log_loss',
        n_jobs=-1,
    )
    imp_df = pd.DataFrame({
        'feature': best['feature_cols'],
        'importance_mean': imp.importances_mean,
        'importance_std': imp.importances_std,
    }).sort_values('importance_mean', ascending=False)
    display(imp_df.head(30))
    plt.figure(figsize=(10, 8))
    top = imp_df.head(20).sort_values('importance_mean')
    plt.barh(top['feature'], top['importance_mean'])
    plt.title('Permutation importance - top 20')
    plt.show()


## 9. Edge threshold tuning

This section only becomes meaningful once you have historical odds for completed games. Until then, it will show sparse or empty results.

In [ ]:
# Join holdout predictions back to market columns from the same games, if present.
market_cols = [c for c in ['game_pk','home_moneyline_median','away_moneyline_median','market_home_no_vig_prob','market_away_no_vig_prob'] if c in completed.columns]
edge_frame = holdout_preds.merge(completed[market_cols], on='game_pk', how='left') if market_cols else holdout_preds.copy()
edge_frame = add_moneyline_edges(edge_frame)
print('Holdout rows with odds:', edge_frame[['home_moneyline_median','away_moneyline_median']].notna().all(axis=1).sum() if {'home_moneyline_median','away_moneyline_median'}.issubset(edge_frame.columns) else 0)
edge_results = tune_moneyline_edge_thresholds(edge_frame)
display(edge_results.head(20))


## 10. Upcoming prediction review

In [ ]:
if PRED_PATH.exists():
    preds = pd.read_csv(PRED_PATH)
    print(preds.shape)
    display(preds.head(30))
else:
    print('No predictions file found yet:', PRED_PATH)


## 11. Export champion model

After reviewing results, set `APPROVE_EXPORT = True` and run this cell. It overwrites:

- `models/mlb_moneyline_champion.joblib`
- `models/mlb_moneyline_champion_metadata.json`

GitHub Actions should load this stable champion artifact for scoring.

In [ ]:
APPROVE_EXPORT = False

if APPROVE_EXPORT:
    metric_row = valid_results[valid_results['candidate_key'] == best_key].iloc[0].to_dict()
    export_paths = export_champion_model(
        estimator=best['estimator'],
        feature_cols=best['feature_cols'],
        metrics={k: v for k, v in metric_row.items() if k.startswith('holdout_') or k in ['cv_log_loss','calibration']},
        model_family=best['model_name'],
        feature_set_name=best['feature_set'],
        model_dir=settings.model_dir,
        notes='Manual champion selected from TetheredAI MLB model lab. No lineup features used.',
    )
    print(export_paths)
else:
    print('Set APPROVE_EXPORT=True after you have reviewed diagnostics.')


## 12. Champion artifact sanity check

In [ ]:
from mlb_betting.modeling import load_model_bundle
champion_path = settings.model_dir / 'mlb_moneyline_champion.joblib'
if champion_path.exists():
    bundle = load_model_bundle(champion_path)
    print('Champion loaded:', champion_path)
    print('Model family:', bundle.get('model_family'))
    print('Feature set:', bundle.get('feature_set_name'))
    print('Feature count:', len(bundle['feature_cols']))
    missing = [c for c in bundle['feature_cols'] if c not in features.columns]
    print('Missing champion features in current feature file:', missing[:25], 'count=', len(missing))
else:
    print('No champion model found yet.')
